In [1]:
import numpy as np
import pandas as pd
import torch
import json
from sklearn.model_selection import KFold
from transformers import BertTokenizer
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset,random_split,SubsetRandomSampler, ConcatDataset
from transformers import BertModel
from torch.optim import Adam
from tqdm import tqdm
from torchmetrics.classification import BinaryStatScores
from datetime import datetime
from collections import Counter
from torcheval.metrics import BinaryAUROC, BinaryAccuracy, BinaryF1Score, BinaryPrecision, BinaryRecall, BinaryConfusionMatrix

In [2]:
labels = torch.load('/mnt/fstore/DataFiles/Saved_Datasets/QALabeling/AMA/PredictedLabels_1.pt')

In [3]:
from tqdm import tqdm
cpu_labels = []
for label in tqdm(labels):
    cpu_labels.append(label.tolist()[0])

100%|██████████| 3319386/3319386 [00:30<00:00, 108586.59it/s]


In [4]:
df = pd.read_pickle('/mnt/fstore/DataFiles/PickledFiles/Data.pkl')
df['Predicted_labels'] = cpu_labels
df['conversation_id'] = df['Hearing']+'_'+df['ID'].astype(str)
df = df.set_index(['conversation_id'])

In [5]:
test_df = pd.read_pickle('/mnt/fstore/DataFiles/PickledFiles/HandLabeledQA.pkl')

In [6]:
print(len(test_df.loc[test_df['Labels'] == 'Questions']), len(test_df.loc[test_df['Labels'] == 'Answers']))

400 412


In [7]:
indices = list(test_df['ID'])
elements = ['CHRG-117shrg45040_66_0','CHRG-117shrg45040_66_1','CHRG-116hhrg37451_60_1','CHRG-116hhrg37451_40_1','CHRG-116hhrg37451_61_1','CHRG-116hhrg37451_62_1','CHRG-116hhrg37451_63_1']
for element in elements:
    indices.remove(element)
test_df = test_df.set_index(['ID'])

In [8]:
test_df = test_df.drop(index=elements)

In [9]:
y_true = []
y_pred = []
LABEL1 = 'Answers'     
LABEL2 = 'Questions' 
labels_dict = {LABEL1:0,LABEL2:1}
for index in indices:
    y_pred.append(df.loc[index,'Predicted_labels'])
    y_true.append(labels_dict[test_df.loc[index,'Labels']])

In [10]:
y_pred = torch.LongTensor(list(y_pred))
y_true = torch.LongTensor(list(y_true))

In [11]:
metric = BinaryConfusionMatrix()
metric.update(y_pred, y_true)
print(f'Confusion Matrix: {metric.compute()}')
metric = BinaryAccuracy()
metric.update(y_pred,y_true)
print(f'Accuracy: {metric.compute()}')
metric = BinaryAUROC()
metric.update(y_pred,y_true)
print(f'AUCROC: {metric.compute()}')
metric = BinaryF1Score()
metric.update(y_pred,y_true)
print(f'F1 Score: {metric.compute()}')
metric = BinaryPrecision()
metric.update(y_pred,y_true)
print(f'Precision: {metric.compute()}')
metric = BinaryRecall()
metric.update(y_pred,y_true)
print(f'Recall: {metric.compute()}')

Confusion Matrix: tensor([[399.,  12.],
        [ 90., 304.]])
Accuracy: 0.8732919096946716
AUCROC: 0.8711882618844714
F1 Score: 0.8563380837440491
Precision: 0.9620253443717957
Recall: 0.7715736031532288


In [12]:
df.head()

,Congress,Hearing,ID,Chamber,Committee,Number,Majority,Role,Speaker,Party,Utterance,Predicted_labels
conversation_id,,,,,,,,,,,,
CHRG-110hhrg36102_1,110,CHRG-110hhrg36102,1,House,Committee on Small Business,1,None,None,None,None,"Mr.Jordan. Good. Thank you. Thank you, Mad...",1
CHRG-110hhrg36102_2,110,CHRG-110hhrg36102,2,House,Committee on Small Business,1,None,None,None,None,ChairwomanBean. Alright. Thank you. Others...,0
CHRG-110hhrg36102_3,110,CHRG-110hhrg36102,3,House,Committee on Small Business,1,None,None,None,None,ChairwomanBean. Absolutely. It wouldn't be...,0
CHRG-110hhrg36102_4,110,CHRG-110hhrg36102,4,House,Committee on Small Business,1,None,None,None,None,ChairwomanBean. Absolutely. Mr. Cochetti? ...,0
CHRG-110hhrg41181_1,110,CHRG-110hhrg41181,1,House,Committee on Financial Services,1,None,None,None,None,The Chairman. The hearing will come to ord...,0


In [13]:
df['ID']

conversation_id
CHRG-110hhrg36102_1      1
CHRG-110hhrg36102_2      2
CHRG-110hhrg36102_3      3
CHRG-110hhrg36102_4      4
CHRG-110hhrg41181_1      1
                        ..
CHRG-112hhrg77036_66    66
CHRG-112hhrg77036_67    67
CHRG-112hhrg77036_68    68
CHRG-112hhrg77036_69    69
CHRG-112hhrg77036_70    70
Name: ID, Length: 3319386, dtype: int64

In [14]:
df['Role'].unique()

array([None, 'member', 'witness'], dtype=object)

## Marking Questions and Answers
Question = 1
Answer = 0

### Logic for is_question:
    1. label must be 1
    2. Party must not be None
    3. Majority must not be None
    4. Speaker must not be None
    3. Role must be member

### Logic for is_answer:
    1. label must be 0
    2. Previous utterance must be labeled 1
    3. Role must be witness
    4. Speaker must not be None

In [8]:
'''Answer = 0, Question = 1'''
#Need to fill: is_question, is_answer, response_to
hearings = list(df['Hearing'].unique())
for hearing in tqdm(hearings):
    indices = list(df.loc[df['Hearing'] == hearing, 'ID'])
    for id in indices:
        index = hearing+'_'+str(id)
        label = df.loc[index,'Predicted_labels']
        if label == 1:
            if df.loc[index,'Party'] is not None and df.loc[index,'Majority'] is not None and df.loc[index, 'Speaker'] is not None and df.loc[index, 'Role'] == 'member':
                df.loc[index,'is_question'] = True
                df.loc[index,'is_answer'] = False
                df.loc[index, 'response_to'] = None
        if label == 0:
            if id == 1:
                continue
            else: 
                prev = hearing+'_'+str(id-1)
            if df.loc[prev,'Predicted_labels'] == 1 and df.loc[index,'Role'] == 'witness' and df.loc[index, 'Speaker'] is not None:
                df.loc[index,'is_question'] = False
                df.loc[index, 'is_answer'] = True
                df.loc[index, 'response_to'] = prev
            

100%|██████████| 17995/17995 [52:30<00:00,  5.71it/s] 


In [12]:
df.to_pickle('/mnt/fstore/DataFiles/PickledFiles/Data_MARK2.pkl')

## Marking the Answer utterances with the party of the questioner

In [10]:
df = pd.read_pickle('/mnt/fstore/DataFiles/PickledFiles/Data_MARK2_withFeatures.pkl')

In [11]:
df.head()

,Congress,Hearing,ID,Chamber,Committee,Number,Majority,Role,Speaker,Party,...,FairnessVice,IngroupVirtue,IngroupVice,AuthorityVirtue,AuthorityVice,PurityVirtue,PurityVice,MoralityGeneral,num_locations,num_dates
conversation_id,,,,,,,,,,,,,,,,,,,,,
CHRG-110hhrg36102_1,110,CHRG-110hhrg36102,1.0,House,Committee on Small Business,1.0,None,None,None,None,...,0.0,0.000913,0.0,0.001826,0.0,0.0,0.0,0.001826,0.0,0.003040
CHRG-110hhrg36102_2,110,CHRG-110hhrg36102,2.0,House,Committee on Small Business,1.0,None,None,None,None,...,0.0,0.000000,0.0,0.003559,0.0,0.0,0.0,0.000000,0.0,0.000000
CHRG-110hhrg36102_3,110,CHRG-110hhrg36102,3.0,House,Committee on Small Business,1.0,None,None,None,None,...,0.0,0.004167,0.0,0.000000,0.0,0.0,0.0,0.004167,0.0,0.000000
CHRG-110hhrg36102_4,110,CHRG-110hhrg36102,4.0,House,Committee on Small Business,1.0,None,None,None,None,...,0.0,0.001890,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.002116
CHRG-110hhrg41181_1,110,CHRG-110hhrg41181,1.0,House,Committee on Financial Services,1.0,None,None,None,None,...,0.0,0.001133,0.0,0.002265,0.0,0.0,0.0,0.001133,0.0,0.008827


In [13]:
for index, row in tqdm(df.loc[df['is_answer']==True].iterrows()):
    df.loc[index,'Party'] = df.loc[row['response_to'],'Party']
    df.loc[index,'Majority'] = df.loc[row['response_to'],'Majority']

677125it [01:39, 6815.78it/s]


In [14]:
df.to_pickle('/mnt/fstore/DataFiles/PickledFiles/Data_MARK2_withFeatures.pkl')

## Making the Confusion Matrix for the Question-Answer Prediction Task

In [1]:
import pandas as pd

In [2]:
df = pd.read_pickle('/mnt/fstore/DataFiles/PickledFiles/Data_MARK2.pkl')

In [3]:
df.columns


Index(['Congress', 'Hearing', 'ID', 'Chamber', 'Committee', 'Number',
       'Majority', 'Role', 'Speaker', 'Party', 'Utterance', 'Predicted_labels',
       'is_question', 'is_answer', 'response_to'],
      dtype='object')

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3319386 entries, CHRG-110hhrg36102_1 to CHRG-112hhrg77036_70
Data columns (total 15 columns):
 #   Column            Dtype 
---  ------            ----- 
 0   Congress          object
 1   Hearing           object
 2   ID                int64 
 3   Chamber           object
 4   Committee         object
 5   Number            int64 
 6   Majority          object
 7   Role              object
 8   Speaker           object
 9   Party             object
 10  Utterance         object
 11  Predicted_labels  int64 
 12  is_question       object
 13  is_answer         object
 14  response_to       object
dtypes: int64(3), object(12)
memory usage: 534.2+ MB


In [8]:
test_df = pd.read_pickle('/mnt/fstore/DataFiles/PickledFiles/HandLabeledQA.pkl')

In [10]:
test_df

,Texts,Labels,ID
0,Mr. Arrington. And who is the head of that...,Questions,CHRG-115hhrg35486_71
1,"Mr. HAGEDORN. Thank you, Madam Chair. Than...",Questions,CHRG-117hhrg45635_57
2,"Mrs. Miller. Thank you, Mr. Chairman. A...",Questions,CHRG-116hhrg36512_31
3,"Mr. LUETKEMEYER. Well, my concern is if we...",Questions,CHRG-115hhrg24421_50
4,"Mr. Lynch. That is a great way to, I think...",Questions,CHRG-115hhrg28505_29
...,...,...,...
807,Mr. Obernolte. Sure. Dr. Jenkins [con...,Answers,CHRG-117hhrg43633_109
808,"Mr. Austin. It is--you know, what I found ...",Answers,CHRG-116hhrg36512_49
809,"Mr. Dodson. First of all, I also want to s...",Answers,CHRG-115hhrg28505_51
810,Dr. Jenkins. So I would say that the bulk...,Answers,CHRG-117hhrg43633_84


In [15]:
confusion_matrix = [[ 0, 1, 2, 3],
                    [ 4, 5, 6, 7],
                    [ 8, 9, 10, 11],
                    [ 12, 13, 14, 15]]

In [28]:
len('CHRG-116hhrg37451_63')

20

In [23]:
int(test_df.loc[0]['ID'][5:8])-114


1

In [30]:
confusion_matrix = [[ 0, 0, 0, 0],[0,0,0,0],[0,0,0,0],[0,0,0,0]]
for _,row in test_df.iterrows():
    if len(row['ID']) >20:
        continue
    if row['Labels'] == 'Questions' and df.loc[row['ID'],'Predicted_labels'] == 1:
        confusion_matrix[0][int(row['ID'][5:8])-114] += 1
    elif row['Labels'] == 'Questions' and df.loc[row['ID'],'Predicted_labels'] == 0:
        confusion_matrix[1][int(row['ID'][5:8])-114] += 1
    elif row['Labels'] == 'Answers' and df.loc[row['ID'],'Predicted_labels'] == 0:
        confusion_matrix[2][int(row['ID'][5:8])-114] += 1
    else:
        confusion_matrix[3][int(row['ID'][5:8])-114] += 1
print(confusion_matrix)

[[20, 105, 101, 65], [6, 25, 29, 28], [26, 127, 134, 93], [0, 3, 3, 5]]
